# CatBoost: pre-bid risk

Run `python preprocessing.py` once from the project folder. The input has one row per auction, and IsBadBuy = 1 is the positive class. Compare the two objectives on the same validation cars using continuous scores.

In [ ]:
import pandas as pd
from preprocessing import CATEGORICAL, NUMERIC, ROOT, load_development
from evaluation import save_validation

X_train, y_train, X_val, y_val = load_development()
scores = pd.DataFrame(index=y_val.index)
print(f"{len(X_train):,} training cars, {len(X_val):,} validation cars")

### Matched boosted-tree settings
Both models receive the same named categorical fields. CatBoost encodes them during fitting. Tree and sampling settings are fixed in advance.

In [ ]:
from catboost import CatBoostClassifier, CatBoostRegressor

settings = dict(iterations=250, depth=6, learning_rate=0.05, l2_leaf_reg=3,
                cat_features=CATEGORICAL, bootstrap_type="Bernoulli", subsample=0.8,
                boosting_type="Plain", one_hot_max_size=2, random_seed=42,
                thread_count=2, verbose=False, allow_writing_files=False)

### Fit squared-error and log-loss versions
RMSE fits the binary outcome as a real-valued target. Keep its raw score for AUC; it need not be a probability.

In [ ]:
cb_l2 = CatBoostRegressor(loss_function="RMSE", **settings)
cb_l2.fit(X_train, y_train)
scores["CatBoost_L2"] = cb_l2.predict(X_val)

cb_log = CatBoostClassifier(loss_function="Logloss", **settings)
cb_log.fit(X_train, y_train)
scores["CatBoost_Logloss"] = cb_log.predict_proba(X_val)[:, 1]

### Validation AUC
Use the shared evaluator and save continuous scores indexed by RefId.

In [ ]:
save_validation(y_val, scores, ROOT / "results" / "validation", "catboost")